# Datamine PLOTPA Process Example

This notebook demonstrates how to configure and run the **`plotpa`** process wrapper in `dmstudio`.

## Process Description

## PLOTPA

Plot all perimeters in a standard perimeter file.

The line type may be chosen by optional parameter. Currently available are broad or narrow. All points are connected in the order in which they appear in the file by the chosen line type. If multiple perimeters exist in the perimeter file, then the required one should be selected by retrieval criteria, e.g. **PVALUE** =3.

### Input Files:

* **in** (Undefined):
  Input data file. This must contain at least two numeric variables ( X and Y) for
  plotting against each other.
  Required=Yes

* **proto** (Plot Prototype):
  The prototype plot file, as input. If this does not contain plot scale information,
  then this must be provided through the optional parameters **XMIN , XMAX , YMIN , YMAX ,

## XSCALE , YSCALE**.

  Required=Yes

* **desc** (Undefined):
  Optional input description file for the case of **ANNOTATE** =3
  Required=No

### Output Files:

* **plot** (Plot):
  Output plot file.
  Required=Yes

### Fields:

* **x** (Numeric : IN):
  The line X co-ordinate.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  The line Y co-ordinate.
  Default=Undefined
  Required=Yes

* **apvalue** (Numeric : IN):
  Field used for **PVALUE** annotation. Default is **PVALUE**.
  Default=Undefined
  Required=No

* **aptn** (Numeric : IN):
  Field used for **PTN** annotation. Default is **PTN**.
  Default=Undefined
  Required=No

* **pcode** (Any : Undefined):
  Optional second perimeter keyfield. The field is of type Alphanumeric and 4 characters.
  Both **PCODE** and **PVALUE** will be annotated by default unless **APVALUE** is
  specified.
  Default=Undefined
  Required=No

* **fields** (Undefined : Undefined):
  First annotation field for **ANNOTATE** =3 case
  Default=Undefined
  Required=No

### Parameters:

* **linecode**:

* **Options** (1: Narrow line.; 2: Broad line.):
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **noclose**:

* **Options** (0: joins the first and last points of the perimeter; =1 does not join the):
  first and last points of the perimeter (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **symbol**:
  Plots the symbol specified at each perimeter point (92). 91=o, 92=+, 93=x.
  Range=Undefined
  Values=Undefined
  Default=92
  Required=No

* **symsize**:
  Symbol size in millimetres (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **annotate**:

* **Options** (0: no annotation; 1: annotate within a break in the perimeter boundary; 2:):
  annotate along the inside of the perimeter boundary; 3: annotate in the 'centre' of the
  perimeter All fields to be plotted must be included in the Fn list taken from the DESC
  file. Default (0).
  Range=0,3
  Values=0,1,2,3
  Default=0
  Required=No

* **ndp**:
  Number of decimal places in the annotation if the annotation field is numeric (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **ndp1**:
  Number of decimal places in the first annotation field **F1**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ndp2**:
  Number of decimal places in the second annotation field **F2**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ndp3**:
  Number of decimal places in the third annotation field **F3**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ndp4**:
  Number of decimal places in the fourth annotation field **F4**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ndp5**:
  Number of decimal places in the fifth annotation field **F5**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ptnannot**:

* **Options** (0: no annotation; =1 annotate the perimeter with the perimeter point number):
  (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **extdis**:
  distance along bisector of perimeter corners for annotation of perimeters. Negative to
  annotate outside the perimeter (4).
  Range=Undefined
  Values=Undefined
  Default=4
  Required=No

* **ptnsize**:
  annotation size for perimeter point number labelling. (2.5)
  Range=Undefined
  Values=Undefined
  Default=2.5
  Required=No

* **charsize**:
  Character size in millimetres (4).
  Range=Undefined
  Values=Undefined
  Default=4
  Required=No

* **aspratio**:
  Aspect ratio, width / ht. for chars (0.9).
  Range=Undefined
  Values=Undefined
  Default=0.9
  Required=No

* **xmin**:
  Minimum value of X for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xmax**:
  Maximum value of X for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ymin**:
  Minimum value of Y for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ymax**:
  Maximum value of Y for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xscale**:
  X scale in user data units per millimetre.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **yscale**:
  Y scale in user data units per millimetre.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('plotpa')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute plotpa
print("Running plotpa...")
dm_cmd.plotpa(
    in_i='t_assays',  # required
    proto_i='t_mod1',  # required
    desc_i='optional',  # required
    plot_o='t_plotpa_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    fields_f=['AU'],  # required
    # apvalue_f='optional',  # optional
    # aptn_f='optional',  # optional
    # pcode_f='optional',  # optional
    # linecode_p=1,  # optional
    # noclose_p=0,  # optional
    # symbol_p=92,  # optional
    # symsize_p=0,  # optional
    # annotate_p=0,  # optional
    # ndp_p=0,  # optional
    # ndps_f=['optional'],  # optional
    # ptnannot_p=0,  # optional
    # extdis_p=4,  # optional
    # ptnsize_p=2.5,  # optional
    # charsize_p=4,  # optional
    # aspratio_p=0.9,  # optional
    # xmin_p='optional',  # optional
    # xmax_p='optional',  # optional
    # ymin_p='optional',  # optional
    # ymax_p='optional',  # optional
    # xscale_p='optional',  # optional
    # yscale_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("plotpa execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_plotpa_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")